# Phase-3 Provenance Audit — seed46 clean single replay

**目的**：在当前 g3_5_5 main HEAD 重跑 `phnode_full clean seed46` 一个 run，判断 catalog 时代的 fragility (epoch 26 起 ODE solver 全 fail，best_loss=0.27, 60s rollout 47 m) 在当前代码上是否仍可复现。

**为什么只跑 seed46**：Phase 2 同口径对齐显示 catalog vs cleanrun v1 的 15.7× gap 由 seed46 (103×) + seed42 (7.3×) 完全驱动；其中 seed46 是训练发散的 catastrophic 失效，是 fragility 最强信号。单跑 seed46 < 5 min，能立即判定 fragility 是否自愈。

**Invocation 链路**：完全复用 `scripts/train_all_models_noise_profile.sh` 与 cleanrun v1 相同的 wrapper，只把 `--seeds` 缩为 "46"，把 `--models` 缩为 "phnode_full"。其它参数严格保留 cleanrun v1 调用形式。

**输出**：`checkpoints/audit_phase3_seed46_clean_<RUN_TAG>/` 一份完整 run（含 training.log / training_history.pkl / best_model.pt / config.json / heldout_eval / rollout_benchmark），打包为 `phase3_seed46_replay_<RUN_TAG>.tar.gz` 下载到本地分析。

**判读**：
- 若 training.log 出现 "no successful training batches"，或 best_epoch < 30，或 60s clean pos_err_median > 10 m → fragility **仍可复现**，Phase 3 进入 Setup B (git bisect)。
- 若 best_epoch ≈ 250 且 60s clean pos_err_median < 1.5 m → fragility 在当前 main 已**自愈**，Phase 3 转 cfg/code swap 归因。

## 0. 环境检查

In [ ]:
!lscpu | head -20

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Project configuration

**重要**：`PROJECT_DIR` 必须指向**当前 g3_5_5 main HEAD 的镜像**（不是 cleanrun v1 时期用的 g3_5_7）。否则跑出来的 fragility 状态属于 cleanrun v1 时代代码，无法回答 "当前 main 是否还发散" 这个问题。

推荐做法：把本地 g3_5_5 仓库（含未 push 的 cx-noise-v3 / provenance-audit-phnode_full 分支也可）同步到 Drive 一份，然后设置 `AUV_PROJECT_DIR` 环境变量指向它。本 notebook 内部已 hard-code 一个候选默认路径，如有不同请改 `os.environ["AUV_PROJECT_DIR"]`。

In [ ]:
import os
from pathlib import Path

# 如果你的 g3_5_5 镜像放在别的位置，改这里：
os.environ.setdefault(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_5",
)

PROJECT_DIR = Path(os.environ["AUV_PROJECT_DIR"])
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

In [ ]:
%pip install -q torchdiffeq pandas

## 2. Git provenance — 记录这次 audit 对应的具体 commit

下面三条命令的输出会被打包随结果一起下载，确保事后能精确知道这次 audit 使用的是哪个 commit。

In [ ]:
!git rev-parse HEAD
!git rev-parse --abbrev-ref HEAD
!git log --oneline -10
!git status --short

## 3. Audit run config

锁定与 cleanrun v1 / catalog 一致的所有 invocation 参数，只把 seeds 缩为 "46"。

Dataset、noise-profile、noise-protocol、noise-reference 与 cleanrun v1 完全相同。`--block-eval-noise-profiles` 与 `--heldout-eval-noise-profiles` 限定为 `clean`，省掉无关 eval profile 节省时间。

In [ ]:
import os
from datetime import datetime

os.environ["AUDIT_RUN_TAG"] = f"audit_phase3_seed46_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.environ["AUDIT_DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["AUDIT_NOISE_REFERENCE"] = "remus100_dr"
os.environ["AUDIT_SUITE_NAME"] = os.environ["AUDIT_RUN_TAG"]
os.environ["AUDIT_DEVICE"] = "cuda"

print("AUDIT_RUN_TAG    =", os.environ["AUDIT_RUN_TAG"])
print("AUDIT_DATASET    =", os.environ["AUDIT_DATASET"])
print("AUDIT_NOISE_REF  =", os.environ["AUDIT_NOISE_REFERENCE"])
print("AUDIT_SUITE_NAME =", os.environ["AUDIT_SUITE_NAME"])
print("AUDIT_DEVICE     =", os.environ["AUDIT_DEVICE"])

## 4. Train: phnode_full × seed46 × clean (~3–5 min Colab T4)

调用同 cleanrun v1 一致的 `train_all_models_noise_profile.sh`，但把 models / seeds 缩到一个 run。所有超参经由 wrapper → `train_auv_hamnode.py` → `DATASET_TRAINING_DEFAULTS["oc"]` 默认值，与 cleanrun v1 完全一致。

In [ ]:
!bash scripts/train_all_models_noise_profile.sh \
  --profile oc \
  --models "phnode_full" \
  --dataset "${AUDIT_DATASET}" \
  --seeds "46" \
  --suite-name "${AUDIT_SUITE_NAME}" \
  --noise-profile clean \
  --noise-protocol auto \
  --noise-reference "${AUDIT_NOISE_REFERENCE}" \
  --block-eval-noise-profiles clean \
  --heldout-eval-noise-profiles clean \
  --device "${AUDIT_DEVICE}"

## 5. 训练 log 关键诊断

三条信号要看：
1. 有没有 "no successful training batches" 行 — 训练发散标志（catalog A46 有 275 行）
2. best epoch / best loss — catalog A46 是 epoch 21 / loss 0.27；cleanrun v1 是 epoch 250 / loss 0.004
3. 完整最后 20 行 — 看 heldout 评估是否完成、有无异常 message

In [ ]:
import subprocess, glob
from pathlib import Path

suite_dir = Path("checkpoints") / os.environ["AUDIT_SUITE_NAME"]
run_dirs = sorted(suite_dir.glob("*phnode_full*seed46*"))
assert run_dirs, f"No phnode_full seed46 run found under {suite_dir}"
RUN_DIR = run_dirs[0]
print("RUN_DIR =", RUN_DIR)

log_path = RUN_DIR / "training.log"
print("\n=== diagnostic 1: count of 'no successful training batches' ===")
nofail = subprocess.run(
    ["grep", "-c", "no successful training batches", str(log_path)],
    capture_output=True, text=True,
)
print("count =", nofail.stdout.strip() or "0")

print("\n=== diagnostic 2: best epoch / best loss (final summary line) ===")
subprocess.run(["grep", "-E", "Best validation score|Done\\.", str(log_path)])

print("\n=== diagnostic 3: last 20 lines ===")
subprocess.run(["tail", "-20", str(log_path)])

In [ ]:
# 额外：扫一下 NaN/Inf/WARNING 出现位置
import subprocess
print("=== any WARNING / ERROR / NaN / Inf in training.log ===")
subprocess.run(["grep", "-nE", "WARNING|ERROR|NaN|Inf|nan", str(log_path)])

## 6. 60s clean rollout — 与 cleanrun v1 / catalog 同协议

调用 `eval_all_models_noise_profile.sh` 在该 suite 上跑 rollout benchmark。limit 在 clean profile + 60s 即可。

In [ ]:
!bash scripts/eval_all_models_noise_profile.sh --suite-dir "checkpoints/${AUDIT_SUITE_NAME}"

## 7. 读出 rollout summary（60s × clean × pos_err_median）

In [ ]:
import json
from pathlib import Path

rb_dirs = sorted((RUN_DIR / "rollout_benchmark").glob("*"))
rb_dirs = [d for d in rb_dirs if d.is_dir() and not d.name.startswith("_")]
print("rollout_benchmark subdirs:")
for d in rb_dirs:
    print(" ", d.name)

# 找 clean profile 的 summary.json
found = []
for d in rb_dirs:
    for sj in d.rglob("clean/summary.json"):
        found.append(sj)
    for sj in d.rglob("summary.json"):
        if sj not in found and "clean" in str(sj):
            found.append(sj)

print("\nclean summary.json candidates:")
for sj in found:
    print(" ", sj)

if found:
    with open(found[0]) as f:
        summary = json.load(f)
    print("\n=== 60s × clean × final_position_error ===")
    try:
        bucket = summary["overall"]["60.0"]["metrics"]["final_position_error"]
        print(json.dumps(bucket, indent=2))
    except Exception as e:
        print("summary schema differs; full dump (first 1500 chars):")
        print(json.dumps(summary, indent=2)[:1500])

## 8. 打包结果用于本地分析

In [ ]:
import subprocess, os
from pathlib import Path

# 记录 git 状态到产物里
audit_meta_dir = Path("checkpoints") / os.environ["AUDIT_SUITE_NAME"] / "_audit_meta"
audit_meta_dir.mkdir(parents=True, exist_ok=True)

with open(audit_meta_dir / "git_provenance.txt", "w") as f:
    for cmd in (
        ["git", "rev-parse", "HEAD"],
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        ["git", "log", "--oneline", "-10"],
        ["git", "status", "--short"],
        ["git", "describe", "--always", "--dirty"],
    ):
        r = subprocess.run(cmd, capture_output=True, text=True)
        f.write("$ " + " ".join(cmd) + "\n")
        f.write(r.stdout)
        if r.stderr:
            f.write("[stderr] " + r.stderr)
        f.write("\n")

# 记录环境
with open(audit_meta_dir / "environment.txt", "w") as f:
    import torch, sys
    f.write(f"python: {sys.version}\n")
    f.write(f"torch:  {torch.__version__}\n")
    f.write(f"cuda:   {torch.version.cuda}\n")
    f.write(f"cudnn:  {torch.backends.cudnn.version()}\n")
    r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    f.write("gpu:\n" + r.stdout + "\n")

print("audit meta files:")
for p in audit_meta_dir.iterdir():
    print(" ", p)

tarball = f"/content/{os.environ['AUDIT_SUITE_NAME']}.tar.gz"
src_dir = f"checkpoints/{os.environ['AUDIT_SUITE_NAME']}"
subprocess.run(["tar", "-czf", tarball, src_dir], check=True)
print("\nwrote tarball:", tarball)
print("size:", subprocess.run(["du", "-h", tarball], capture_output=True, text=True).stdout.strip())

In [ ]:
# 触发浏览器下载（如果你想直接保存到 Drive，把 tarball 拷到 Drive 也行）
from google.colab import files
files.download(tarball)

## 9. 决策矩阵（本地分析时参照）

下载到本地后放进 `analysis/provenance_audit/phase3_retrain/`：

```bash
mkdir -p analysis/provenance_audit/phase3_retrain
tar -xzf audit_phase3_seed46_clean_<RUN_TAG>.tar.gz -C analysis/provenance_audit/phase3_retrain
```

判读规则：

| 信号 | 触发条件 | 解读 | 下一步 |
| --- | --- | --- | --- |
| **A** | training.log 含 "no successful training batches" ≥ 1 行 | fragility 在当前 main 仍可复现 | Phase 3 Setup B：git bisect 在 cleanrun v1 ↔ 当前 HEAD 之间找修复发散的 commit |
| **B** | best_epoch ≥ 200 且 60s clean pos_err_median < 1.5 m | fragility 已自愈 | Phase 3 简化为 cfg/code path 单变量 swap，定位自愈来源 |
| **C** | best_epoch ∈ [30, 200) 或 60s pos_err_median ∈ [1.5, 10) m | 部分发散 / 半自愈 | 复跑 1 次确认是否随 CUDA 非确定性波动；再决定 |
| **D** | rollout 出错 / eval 未完成 | invocation 出问题 | 检查 git status 是否含未提交改动，或 DATASET 路径是否对 |